# 🚀 Freqtrade RL Trading Validation - Google Colab

این نوتبوک برای اجرای بک‌تست استراتژی `MtfScalper_RL_Hybrid` در محیط Google Colab طراحی شده است.

## 📋 مراحل:
1. نصب TA-Lib (پیش‌نیاز)
2. نصب Freqtrade و وابستگی‌ها
3. کلون ریپازیتوری
4. بررسی داده‌های Multi-Timeframe
5. اجرای بک‌تست
6. تحلیل نتایج
7. دانلود داده‌های آنالیز

## 🔧 مرحله 1: نصب TA-Lib (پیش‌نیاز مهم)

TA-Lib باید قبل از Freqtrade نصب شود چون نیاز به کامپایل دارد.

In [ ]:
# نصب TA-Lib از طریق apt (برای Ubuntu/Debian)
!apt-get update
!apt-get install -y build-essential wget

# دانلود و نصب TA-Lib از سورس
!wget http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
!cd ta-lib && ./configure --prefix=/usr && make && make install

# نصب wrapper Python برای TA-Lib
!pip install TA-Lib

## 📦 مرحله 2: نصب Freqtrade و وابستگی‌های اصلی

In [ ]:
# نصب Freqtrade با FreqAI support
!pip install freqtrade[freqai]

# نصب PyTorch (برای GPU support در Colab)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

# نصب Stable-Baselines3 برای RL
!pip install stable-baselines3>=2.2.0 gymnasium>=0.28.0

# نصب کتابخانه‌های تحلیل داده
!pip install pandas>=2.0.3 numpy>=1.26.0 scikit-learn>=1.3.0

# نصب ابزارهای visualization
!pip install matplotlib seaborn plotly

# نصب سایر وابستگی‌ها
!pip install datasieve scipy joblib tqdm pytz

## 📥 مرحله 3: کلون ریپازیتوری و Checkout برنچ

In [ ]:
# پاک کردن دایرکتوری قبلی (اگر وجود دارد)
!rm -rf RL-Trading-Validation

# کلون ریپازیتوری
!git clone https://github.com/aminak58/RL-Trading-Validation.git

# تغییر دایرکتوری
%cd RL-Trading-Validation

# Checkout برنچ مورد نظر
!git checkout fix/signal-features-variance-threshold

# نمایش آخرین کامیت
!git log -1 --oneline

## 🔍 مرحله 4: بررسی داده‌های Multi-Timeframe

استراتژی `MtfScalper_RL_Hybrid` از چند تایم‌فریم استفاده می‌کند:
- **1m**: تایم‌فریم اصلی
- **5m, 15m, 1h**: تایم‌فریم‌های اطلاعاتی
- **4h**: ترند بلندمدت
- **8h-mark, 8h-funding_rate**: داده‌های فیوچرز

In [ ]:
# لیست فایل‌های داده
!ls -lh user_data/data/binance/futures/

# بررسی وجود تمام فایل‌های مورد نیاز
import os

required_files = [
    "user_data/data/binance/futures/BTC_USDT_USDT-1m-futures.feather",
    "user_data/data/binance/futures/BTC_USDT_USDT-5m-futures.feather",
    "user_data/data/binance/futures/BTC_USDT_USDT-15m-futures.feather",
    "user_data/data/binance/futures/BTC_USDT_USDT-1h-futures.feather",
    "user_data/data/binance/futures/BTC_USDT_USDT-4h-futures.feather",
    "user_data/data/binance/futures/BTC_USDT_USDT-8h-mark.feather",
    "user_data/data/binance/futures/BTC_USDT_USDT-8h-funding_rate.feather",
]

print("\n📊 بررسی فایل‌های Multi-Timeframe:\n")
all_present = True
total_size_mb = 0

for data_file in required_files:
    if os.path.exists(data_file):
        size_mb = os.path.getsize(data_file) / (1024 * 1024)
        total_size_mb += size_mb
        filename = os.path.basename(data_file)
        print(f"✅ {filename:45s} - {size_mb:8.2f} MB")
    else:
        filename = os.path.basename(data_file)
        print(f"❌ {filename:45s} - NOT FOUND")
        all_present = False

print(f"\n{'='*60}")
if all_present:
    print(f"✅ همه فایل‌ها موجود هستند (مجموع: {total_size_mb:.2f} MB)")
else:
    print("❌ برخی فایل‌ها یافت نشدند! لطفاً داده‌ها را دانلود کنید.")

## 🔍 مرحله 5: بررسی کانفیگ و استراتژی

In [ ]:
# نمایش کانفیگ اصلی
!cat configs/config_rl_hybrid.json | head -50

# بررسی وجود فایل‌های استراتژی
!ls -lh user_data/strategies/
!ls -lh user_data/freqaimodels/

## 🎯 مرحله 6: اجرای بک‌تست

**توجه**: این مرحله ممکن است 20-30 دقیقه طول بکشد (بسته به GPU).

In [ ]:
# اجرای بک‌تست با لاگ کامل
!freqtrade backtesting \
    --config configs/config_rl_hybrid.json \
    --strategy MtfScalper_RL_Hybrid \
    --freqaimodel MtfScalperRLModel \
    --timerange 20241001-20241007 \
    --breakdown day \
    2>&1 | tee test_week_fix.log

## 📊 مرحله 7: نمایش نتایج

In [ ]:
# نمایش 100 خط آخر لاگ
!tail -n 100 test_week_fix.log

In [ ]:
# جستجوی خطاها در لاگ
!grep -i "error\|exception\|failed" test_week_fix.log | tail -20

In [ ]:
# نمایش نتایج backtest (اگر موفق بود)
!ls -lh user_data/backtest_results/

## 📈 مرحله 8: بررسی داده‌های آنالیز RL

این داده‌ها توسط `DataCollector` جمع‌آوری می‌شوند و برای تحلیل عمیق رفتار مدل RL حیاتی هستند.

In [ ]:
# بررسی فایل‌های آنالیز
!ls -lh user_data/analysis_data/rl_training/

# نمایش خلاصه آنالیز (اگر موجود باشد)
import json
from pathlib import Path

analysis_dir = Path("user_data/analysis_data/rl_training")
summary_files = list(analysis_dir.glob("summary_*.json"))

if summary_files:
    latest_summary = sorted(summary_files)[-1]
    print(f"\n📊 خلاصه آنالیز: {latest_summary.name}\n")
    
    with open(latest_summary, 'r') as f:
        summary = json.load(f)
    
    # نمایش تعداد داده‌ها
    if 'data_counts' in summary:
        print("📦 تعداد داده‌های جمع‌آوری شده:")
        for key, value in summary['data_counts'].items():
            print(f"  - {key}: {value}")
    
    # نمایش آمار معاملات
    if 'trade_stats' in summary:
        print("\n💰 آمار معاملات:")
        stats = summary['trade_stats']
        print(f"  - تعداد کل: {stats['total_trades']}")
        print(f"  - نرخ برد: {stats['win_rate']:.2%}")
        print(f"  - میانگین سود: {stats['avg_profit']:.2%}")
        print(f"  - Profit Factor: {stats['profit_factor']:.2f}")
    
    # نمایش آمار RL
    if 'rl_stats' in summary:
        print("\n🤖 آمار RL:")
        stats = summary['rl_stats']
        print(f"  - تعداد اپیزودها: {stats['total_episodes']}")
        print(f"  - میانگین reward: {stats['avg_episode_reward']:.2f}")
        print(f"  - بهترین episode: {stats['max_episode_reward']:.2f}")
else:
    print("⚠️ هیچ فایل آنالیزی یافت نشد.")

## 🐛 مرحله 9: دیباگ (در صورت بروز خطا)

اگر خطایی رخ داد، سلول‌های زیر را اجرا کنید:

In [ ]:
# بررسی نسخه‌های نصب شده
!pip show freqtrade stable-baselines3 torch gymnasium pandas

In [ ]:
# بررسی GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# تست import استراتژی
import sys
sys.path.insert(0, '/content/RL-Trading-Validation')

try:
    from user_data.strategies.MtfScalper_RL_Hybrid import MtfScalper_RL_Hybrid
    print("✅ استراتژی با موفقیت import شد")
except Exception as e:
    print(f"❌ خطا در import استراتژی: {e}")

try:
    from user_data.freqaimodels.MtfScalperRLModel import MtfScalperRLModel
    print("✅ مدل RL با موفقیت import شد")
except Exception as e:
    print(f"❌ خطا در import مدل RL: {e}")

## 💾 مرحله 10: دانلود نتایج و داده‌های آنالیز

**مهم**: داده‌های `analysis_data` برای تحلیل عمیق رفتار مدل RL حیاتی هستند و شامل:
- **trades_*.json/csv**: اطلاعات کامل هر معامله
- **rl_episodes_*.json**: داده‌های آموزش RL
- **reward_breakdown_*.json**: تحلیل جزئیات reward function
- **signal_propagation_*.json**: ردیابی سیگنال‌ها از تولید تا اجرا
- **model_decisions_*.json**: تصمیمات مدل و feature importance
- **pipeline_breakdowns_*.json**: نقاط شکست در pipeline

In [ ]:
# فشرده‌سازی نتایج کامل
!tar -czf backtest_results_complete.tar.gz \
    test_week_fix.log \
    user_data/backtest_results/ \
    user_data/models/ \
    user_data/analysis_data/

# نمایش حجم فایل فشرده
!ls -lh backtest_results_complete.tar.gz

In [ ]:
# دانلود (در Colab)
from google.colab import files
files.download('backtest_results_complete.tar.gz')

## 📊 مرحله 11: آنالیز سریع داده‌ها (اختیاری)

تحلیل سریع داده‌های جمع‌آوری شده قبل از دانلود:

In [ ]:
# استفاده از DataCollector برای آنالیز
from user_data.data_collector import analyze_session

# آنالیز آخرین session
analyze_session(data_dir="user_data/analysis_data/rl_training")

In [ ]:
# نمایش reward breakdown (اگر موجود باشد)
import pandas as pd
from pathlib import Path

analysis_dir = Path("user_data/analysis_data/rl_training")
reward_files = list(analysis_dir.glob("reward_breakdown_*.csv"))

if reward_files:
    latest_rewards = sorted(reward_files)[-1]
    df_rewards = pd.read_csv(latest_rewards)
    
    print(f"\n📊 تحلیل Reward Breakdown ({len(df_rewards)} رکورد):\n")
    
    # میانگین هر component
    if 'components' in df_rewards.columns:
        print("میانگین component های reward:")
        # Note: components might be stored as JSON string
        print(df_rewards[['reward_type', 'raw_reward', 'normalized_reward']].describe())
else:
    print("⚠️ فایل reward breakdown یافت نشد.")